# Lecture 7 · Retrieval-augmented generation (Fable edition)

Companion notebook, generated from the code on the lecture pages (https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-07-fable/). Run the cells in order; each one corresponds to a numbered step on the pages. Steps marked 💳 need `LLM_API_KEY` (and optionally `LLM_BASE_URL`, `LLM_MODEL`) in the environment and are skipped otherwise.

## Step 1 — Ask a plain model 💳 API

[1. A question the model cannot answer](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-07-fable/01-a-question-the-model-cannot-answer/)

In [ ]:
import os
from openai import OpenAI

# Three environment variables select the provider; Section 5.2 explains them.
#   LLM_API_KEY   your key (OpenAI, OpenRouter, ...)
#   LLM_BASE_URL  optional; leave unset for OpenAI
#   LLM_MODEL     a cheap chat model, e.g. "gpt-4o-mini"
# steps marked 💳 are skipped when this is False
LLM_READY = bool(os.environ.get("LLM_API_KEY"))
client = OpenAI(api_key=os.environ["LLM_API_KEY"],
                base_url=os.environ.get("LLM_BASE_URL")) if LLM_READY else None
MODEL = os.environ.get("LLM_MODEL", "gpt-4o-mini")

def ask_llm(prompt: str, system: str = "You are a precise assistant.") -> str:
    assert LLM_READY, ("set LLM_API_KEY (and optionally LLM_BASE_URL, "
                       "LLM_MODEL) before running this step")
    r = client.chat.completions.create(
        model=MODEL,
        temperature=0,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": prompt}],
    )
    return r.choices[0].message.content.strip()

QUESTION = (
    "In the study notes of the course 'Generative and Agentic AI', Lecture 4, "
    "which worked example illustrates Least-to-Most prompting, "
    "and what are its three subquestions?")
if LLM_READY:
    print(ask_llm(QUESTION))

## Step 2 — Load the corpus

[2. Build a retriever by hand](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-07-fable/02-build-a-retriever-by-hand/)

In [ ]:
import json, urllib.request, pathlib

CORPUS_URL = ("https://aiml-study-guides.onrender.com/corpus/generative-"
              "agentic-ai/lectures-01-06.en.jsonl")
path = pathlib.Path("lectures-01-06.en.jsonl")
if not path.exists():
    urllib.request.urlretrieve(CORPUS_URL, path)

sections = [json.loads(line) for line in path.open(encoding="utf-8")]
print(len(sections), "sections,", sum(s["words"] for s in sections), "words")

s = sections[40]
print({k: s[k] for k in ("id", "lecture", "page", "heading", "url", "words")})
print(s["text"][:300], "…")

## Step 3 — Split sections into chunks

[2. Build a retriever by hand](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-07-fable/02-build-a-retriever-by-hand/)

In [ ]:
import re

def split_text(text: str, size: int = 500, overlap: int = 100) -> list[str]:
    """Recursive character splitting: cut at paragraph breaks if possible,
    then line breaks, then sentence ends, then spaces; merge pieces up to
    `size`."""
    if len(text) <= size:
        return [text]
    for sep in ["\n\n", "\n", r"(?<=[.!?])\s+", " "]:
        pieces = [p for p in re.split(sep, text) if p.strip()]
        if len(pieces) < 2:
            continue
        glue = " " if sep.startswith("(") else sep
        chunks, current = [], []
        for piece in pieces:
            if len(piece) > size:  # a piece that is itself too long: recurse
                if current:
                    chunks.append(glue.join(current)); current = []
                chunks.extend(split_text(piece, size, overlap))
                continue
            if current and len(glue.join(current + [piece])) > size:
                chunks.append(glue.join(current))
                tail, kept = [], 0  # overlap: keep trailing pieces that fit
                for p in reversed(current):
                    if kept + len(p) > overlap:
                        break
                    tail.insert(0, p); kept += len(p) + len(glue)
                # never exceed `size`
                while tail and len(glue.join(tail + [piece])) > size:
                    tail.pop(0)
                current = tail
            current.append(piece)
        if current:
            chunks.append(glue.join(current))
        return chunks
    return [text[i:i + size] for i in range(0, len(text), size - overlap)]

chunks = []
for s in sections:
    for i, piece in enumerate(split_text(s["text"], size=500, overlap=100)):
        chunks.append({
            "id": f'{s["id"]}~{i}', "text": piece, "section": s["id"],
            "page": s["page"], "lecture": s["lecture"],
            "heading": s["heading"], "url": s["url"]
        })

print(len(chunks), "chunks from", len(sections), "sections")
lengths = [len(c["text"]) for c in chunks]
print("characters per chunk: min", min(lengths), "median",
      sorted(lengths)[len(lengths) // 2], "max", max(lengths))
print(chunks[100]["id"]); print(chunks[100]["text"])

## Step 4 — Score chunks by their words: BM25

[2. Build a retriever by hand](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-07-fable/02-build-a-retriever-by-hand/)

In [ ]:
import math
from collections import Counter

def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())

class BM25:
    def __init__(self, docs: list[str], k1: float = 1.5, b: float = 0.75):
        self.k1, self.b = k1, b
        self.tokens = [tokenize(d) for d in docs]
        self.tf = [Counter(t) for t in self.tokens]
        self.doc_len = [len(t) for t in self.tokens]
        self.avg_len = sum(self.doc_len) / len(self.doc_len)
        df = Counter(term for t in self.tokens for term in set(t))
        n = len(docs)
        self.idf = {
            term: math.log(1 + (n - d + 0.5) / (d + 0.5))
            for term, d in df.items()
        }

    def score(self, query: str, i: int) -> float:
        total = 0.0
        for term in tokenize(query):
            f = self.tf[i].get(term, 0)
            if not f:
                continue
            norm = 1 - self.b + self.b * self.doc_len[i] / self.avg_len
            total += self.idf[term] * f * (self.k1 + 1) / (f + self.k1 * norm)
        return total

    def search(self, query: str, k: int = 5) -> list[tuple[float, int]]:
        scored = [(self.score(query, i), i) for i in range(len(self.tokens))]
        return sorted(scored, reverse=True)[:k]

bm25 = BM25([c["text"] for c in chunks])

def show(results, chunks=chunks):
    for score, i in results:
        c = chunks[i]
        print(f'{score:6.2f}  L{c["lecture"]} · {c["heading"][:60]}')
        print("        ", c["text"][:140].replace("\n", " "), "…")

show(
    bm25.search(
        "Which worked example illustrates Least-to-Most prompting and what "
        "are its subquestions?"))

## Step 5 — Where words fail

[2. Build a retriever by hand](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-07-fable/02-build-a-retriever-by-hand/)

In [ ]:
for q in ["how does the model decide which tool to call",
          "what stops the model from producing endless text",
          "why do models make things up"]:
    print("\n", q)
    show(bm25.search(q, k=3))

## Step 6 — Turn text into vectors

[3. Measure meaning with vectors](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-07-fable/03-measure-meaning-with-vectors/)

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# English; see 3.4 for a multilingual choice
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL)

texts = [
    "why do models make things up",
    "a hallucination is generated information presented as though "
    "it were grounded", "the shop had 30 headphones and sold 9"
]
vectors = embedder.encode(texts, normalize_embeddings=True)
print(vectors.shape)  # (3, 384): three texts, 384 numbers each
# the first eight numbers of the first vector
print(np.round(vectors[0][:8], 3), "…")

## Step 7 — Compare vectors by hand

[3. Measure meaning with vectors](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-07-fable/03-measure-meaning-with-vectors/)

In [ ]:
def dot(a, b):        return float(np.sum(a * b))
def length(a):        return float(np.sqrt(np.sum(a * a)))
def cosine(a, b):     return dot(a, b) / (length(a) * length(b))
def euclidean(a, b):  return length(a - b)

q, h, shop = vectors
for name, v in [("hallucination sentence", h), ("headphone sentence", shop)]:
    print(f"{name:24s} dot={dot(q, v):.3f}  cosine={cosine(q, v):.3f}  "
          f"distance={euclidean(q, v):.3f}")
print("lengths:", round(length(q), 3), round(length(h), 3),
      round(length(shop), 3))

## Step 8 — Embed the corpus and search by meaning

[3. Measure meaning with vectors](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-07-fable/03-measure-meaning-with-vectors/)

In [ ]:
import time
t0 = time.time()
chunk_vectors = embedder.encode([c["text"] for c in chunks],
                                normalize_embeddings=True, batch_size=64)
print(chunk_vectors.shape, f"embedded in {time.time() - t0:.0f} s")

def vector_search(query: str, k: int = 5):
    qv = embedder.encode([query], normalize_embeddings=True)[0]
    # one dot product per chunk = cosine, since all are unit length
    scores = chunk_vectors @ qv
    top = np.argsort(-scores)[:k]
    return [(float(scores[i]), int(i)) for i in top]

for q in ["how does the model decide which tool to call",
          "what stops the model from producing endless text",
          "why do models make things up"]:
    print("\n", q)
    show(vector_search(q, k=3))

## Step 9 — A first measurement: recall@5 of both retrievers

[3. Measure meaning with vectors](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-07-fable/03-measure-meaning-with-vectors/)

In [ ]:
# (question, acceptable section ids — the part of the id after
# "en/generative-agentic-ai/")
TESTS = [
    ("Which worked example illustrates Least-to-Most prompting?", [
        "lecture-04/03-choose-a-reasoning-or-tool-workflow#32-decomposition-"
        "least-to-most-and-plan-and-solve"
    ]),
    ("how does the model decide which tool to call", [
        "lecture-06/01-decide-what-the-application-should-control#11-from-an-"
        "answer-to-an-action"
    ]),
    ("what stops the model from producing endless text", [
        "lecture-03/04-configure-generation#44-output-limits-stopping-and-"
        "valid-output-formats",
        "lecture-01/02-text-generation-autoregression-and-attention#21-"
        "predict-append-repeat"
    ]),
    ("why do models make things up", [
        "lecture-01/01-models-and-learning-foundations#15-models-as-input-"
        "output-components",
        "lecture-04/04-make-answers-checkable#41-factual-source-instruction-"
        "and-reasoning-failures"
    ]),
    ("what does temperature change about sampling", [
        "lecture-03/04-configure-generation#42-temperature-and-the-"
        "distribution-of-choices"
    ]),
    ("difference between training and inference", [
        "lecture-01/01-models-and-learning-foundations"
        "#12-training-inference-and-adaptation"
    ]),
    ("accuracy drops when the relevant passage sits in the middle of "
     "a long prompt", [
         "lecture-03/02-check-practical-constraints#22-context-and-output-"
         "limits-and-effective-use-of-long-documents",
         "lecture-04/02-use-examples-and-manage-limited-context#23-long-"
         "documents-relevant-passages-history-and-memory"
     ]),
    ("how does DPO differ from RLHF", [
        "lecture-02/03-training-a-useful-assistant#33-preference-training-"
        "rlhf-dpo-and-orpo"
    ]),
    ("what is latent diffusion", [
        "lecture-01/03-image-generation-diffusion"
        "#33-latent-diffusion-and-image-decoding"
    ]),
    ("validate the model's output with pydantic", [
        "lecture-05/04-return-data-that-application-code-can-use#41-define-"
        "and-validate-an-output-with-pydantic"
    ]),
    ("what is indirect prompt injection", [
        "lecture-04/05-keep-external-content-from-taking-control#52-direct-"
        "indirect-multimodal-and-many-shot-attacks"
    ]),
    ("how many training tokens per parameter does Chinchilla recommend", [
        "lecture-02/01-measuring-learning-and-planning-training#14-"
        "allocating-a-fixed-compute-budget-chinchilla",
        "lecture-03/02-check-practical-constraints#23-parameter-count-"
        "training-quality-and-resource-requirements"
    ]),
    ("cheapest way to make a model fit on a smaller GPU", [
        "lecture-02/04-making-the-model-practical-to-run#42-quantization-"
        "representing-numbers-with-fewer-bits"
    ]),
    ("see every step of a request in LangSmith", [
        "lecture-05/06-inspect-test-and-deploy-the-application#61-langsmith-"
        "follow-a-request-through-its-steps"
    ]),
    ("what is a mixture of experts model", [
        "lecture-03/03-design-choices-that-affect-the-trade-offs#31-mixture-"
        "of-experts-total-versus-active-parameters"
    ]),
    ("top-p versus top-k", [
        "lecture-03/04-configure-generation"
        "#43-top-k-and-top-p-candidate-filtering"
    ]),
]
PREFIX = "en/generative-agentic-ai/"

def recall_at_k(search, k=5):
    hits = 0
    for question, gold in TESTS:
        found = {chunks[i]["section"] for _, i in search(question, k)}
        hits += any(PREFIX + g in found for g in gold)
    return hits / len(TESTS)

print(f"BM25    recall@5 = {recall_at_k(bm25.search):.2f}")
print(f"vectors recall@5 = {recall_at_k(vector_search):.2f}")
for question, gold in TESTS:
    b = any(PREFIX +
            g in {chunks[i]["section"]
                  for _, i in bm25.search(question, 5)} for g in gold)
    v = any(PREFIX +
            g in {chunks[i]["section"]
                  for _, i in vector_search(question, 5)} for g in gold)
    print(f'{"✓" if b else "·"} BM25  {"✓" if v else "·"} vec   {question}')

## Step 10 — A vector store with metadata: Chroma

[4. Store and search at scale](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-07-fable/04-store-and-search-at-scale/)

In [ ]:
import chromadb

# in memory; chromadb.PersistentClient("chroma_db") keeps it on disk
chroma = chromadb.Client()
if "course_notes" in [c.name for c in chroma.list_collections()]:
    # so that re-running this step starts from an empty collection
    chroma.delete_collection("course_notes")
collection = chroma.create_collection(
    "course_notes", configuration={"hnsw": {"space": "cosine"}})
collection.add(
    ids=[c["id"] for c in chunks],
    documents=[c["text"] for c in chunks],
    # we already computed them; Chroma can also call an embedding function
    # itself
    embeddings=chunk_vectors.tolist(),
    metadatas=[{
        "lecture": c["lecture"], "section": c["section"],
        "heading": c["heading"], "url": c["url"]
    } for c in chunks],
)
print(collection.count(), "chunks stored")

def chroma_search(query: str, k: int = 5, where: dict | None = None):
    qv = embedder.encode([query], normalize_embeddings=True).tolist()
    res = collection.query(query_embeddings=qv, n_results=k, where=where)
    return [(1 - dist, meta, doc) for dist, meta, doc in zip(
        res["distances"][0], res["metadatas"][0], res["documents"][0])]

question = "how is a long task split into smaller steps"
for where in (None, {"lecture": 4}, {"lecture": {"$gte": 5}}):
    print("\nfilter:", where)
    for sim, meta, doc in chroma_search(question, k=3, where=where):
        print(f'{sim:.3f}  L{meta["lecture"]} '
              f'· {meta["heading"][:42]:42s} {doc[:44]!r}…')

## Step 11 — Combine keyword and vector search with reciprocal rank fusion

[4. Store and search at scale](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-07-fable/04-store-and-search-at-scale/)

In [ ]:
def rrf(*rankings, k: int = 60):
    """Reciprocal rank fusion: each list contributes 1/(k + rank) per item;
    sum across lists."""
    scores = {}
    for ranking in rankings:
        for rank, (_, i) in enumerate(ranking, start=1):
            scores[i] = scores.get(i, 0.0) + 1.0 / (k + rank)
    return sorted(((s, i) for i, s in scores.items()), reverse=True)

def hybrid_search(query: str, k: int = 5):
    return rrf(bm25.search(query, 20), vector_search(query, 20))[:k]

show(bm25.search("what does the lookup_demo_order tool return", k=3)); print()
show(vector_search("what does the lookup_demo_order tool return", k=3))
print()
show(hybrid_search("what does the lookup_demo_order tool return", k=3))
print()
for name, fn in [("BM25", bm25.search), ("vectors", vector_search),
                 ("hybrid", hybrid_search)]:
    print(f"{name:8s} recall@5 = {recall_at_k(fn):.2f}")

## Step 12 — Assemble the context

[5. Generate a grounded answer](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-07-fable/05-generate-a-grounded-answer/)

In [ ]:
def build_context(results, max_chars: int = 3000):
    """Number the passages, label each with its provenance, stop at the
    character budget."""
    blocks, sources, used = [], [], 0
    for n, (_, i) in enumerate(results, start=1):
        c = chunks[i]
        block = (f"[{n}] Lecture {c['lecture']} · "
                 f"{c['heading'] or c['page']}\n{c['text']}")
        if used + len(block) > max_chars:
            break
        blocks.append(block); sources.append(c); used += len(block)
    return "\n\n".join(blocks), sources

context, sources = build_context(
    hybrid_search("Which worked example illustrates Least-to-Most prompting?",
                  k=5))
print(len(context), "characters,", len(sources), "passages")
print(context[:600], "…")

## Step 13 — Prompt for an answer with citations 💳 API

[5. Generate a grounded answer](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-07-fable/05-generate-a-grounded-answer/)

In [ ]:
SYSTEM = ("You answer questions about a set of course notes. Use only the "
          "numbered sources provided.")
TEMPLATE = """Sources:
{context}

Question: {question}

Rules:
- Answer only from the sources above. Do not use outside knowledge.
- After each claim, cite the source number in square brackets, e.g. [2].
- If the sources do not contain the answer, reply exactly: \
"Not in the notes." and nothing else.
Answer in at most four sentences."""

def rag_answer(question: str, k: int = 5):
    context, sources = build_context(hybrid_search(question, k))
    answer = ask_llm(TEMPLATE.format(context=context, question=question),
                     system=SYSTEM)
    cited = sorted({int(n) for n in re.findall(r"\[(\d+)\]", answer)})
    links = [
        f"[{n}] {sources[n-1]['url']}" for n in cited if 1 <= n <= len(sources)
    ]
    return answer, links

if LLM_READY:
    for q in [
            "Which worked example illustrates Least-to-Most prompting, and "
            "what are its three subquestions?",
            "What does the course say about deploying with Kubernetes?"
    ]:
        answer, links = rag_answer(q)
        print("\nQ:", q); print("A:", answer); print("\n".join(links))

## Step 14 — The same pipeline in LangChain (💳 API for the final call)

[5. Generate a grounded answer](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-07-fable/05-generate-a-grounded-answer/)

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

docs = [
    Document(
        page_content=s["text"], metadata={
            "lecture": s["lecture"], "heading": s["heading"], "url": s["url"]
        }) for s in sections
]
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = splitter.split_documents(docs)  # Step 3, as a library call
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)   # Step 6
vectorstore = Chroma.from_documents(
    splits,
    embeddings,
    collection_name="course_notes_lc",
    # Steps 8 + 10
    collection_configuration={"hnsw": {"space": "cosine"}})
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
print(len(splits), "splits;",
      len(retriever.invoke("what is latent diffusion")), "retrieved")

prompt = ChatPromptTemplate.from_messages([("system", SYSTEM),
                                           ("human", TEMPLATE)])
llm = ChatOpenAI(model=MODEL, temperature=0,
                 api_key=os.environ.get("LLM_API_KEY", "unset"),
                 base_url=os.environ.get("LLM_BASE_URL"))

def format_docs(docs):
    return "\n\n".join(f"[{n}] Lecture {d.metadata['lecture']} · "
                       f"{d.metadata['heading']}\n{d.page_content}"
                       for n, d in enumerate(docs, start=1))

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
if LLM_READY:
    print(rag_chain.invoke("What is latent diffusion?"))

## Step 15 — Measure retrieval and answers separately (💳 API for the answer part)

[5. Generate a grounded answer](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-07-fable/05-generate-a-grounded-answer/)

In [ ]:
def mrr(search, k: int = 10):
    """Mean reciprocal rank: 1/position of the first correct section,
    averaged over questions."""
    total = 0.0
    for question, gold in TESTS:
        for rank, (_, i) in enumerate(search(question, k), start=1):
            if any(chunks[i]["section"] == PREFIX + g for g in gold):
                total += 1 / rank
                break
    return total / len(TESTS)

for name, fn in [("BM25", bm25.search), ("vectors", vector_search),
                 ("hybrid", hybrid_search)]:
    print(
        f"{name:8s} recall@5 = {recall_at_k(fn):.2f}   MRR@10 = {mrr(fn):.2f}")

# (question, facts the answer must contain, must the system refuse?)
ANSWER_TESTS = [
    ("Which worked example illustrates Least-to-Most prompting?",
     ["headphone", "boxes"], False),
    ("In the Lecture 4 shop example, how many headphones are available after "
     "the sale and the delivery?", ["33"], False),
    ("What does the course say about deploying with Kubernetes?", [], True),
]
JUDGE = """Question: {question}
Reference facts the answer must contain: {facts}
Candidate answer: {answer}
Does the candidate contain every reference fact, without contradicting \
them? Reply YES or NO."""

if LLM_READY:
    for question, facts, must_refuse in ANSWER_TESTS:
        answer, _ = rag_answer(question)
        refused = answer.strip().startswith("Not in the notes")
        if must_refuse:
            verdict = ("refused ✓" if refused
                       else "answered ✗ (should have refused)")
        elif refused:
            verdict = "refused ✗ (answer exists)"
        else:
            verdict = "facts " + ask_llm(
                JUDGE.format(question=question, facts=facts, answer=answer))
        print(f"{verdict:34s} {question}")

## Step 16 — Multi-Query and RAG-Fusion 💳 API

[6. When the question is the problem](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-07-fable/06-when-the-question-is-the-problem/)

In [ ]:
def rewrite(question: str, n: int = 3) -> list[str]:
    text = ask_llm(
        f"Write {n} alternative search queries for the question below, "
        f"each using different "
        f"vocabulary. One query per line, no numbering, no "
        f"explanations.\n\nQuestion: {question}")
    variants = [
        line.strip("-•* ").strip() for line in text.splitlines()
        if line.strip()
    ]
    return [question] + variants[:n]

# union, taking each phrasing's hits in turn
def multi_query_search(question: str, k: int = 5):
    lists = [hybrid_search(q, k) for q in rewrite(question)]
    seen, out = set(), []
    for rank in range(k):
        for hits in lists:
            if rank < len(hits) and hits[rank][1] not in seen:
                seen.add(hits[rank][1]); out.append(hits[rank])
    return out[:k]

def fusion_search(question: str, k: int = 5):  # RAG-Fusion: RRF over the lists
    return rrf(*[hybrid_search(q, 20) for q in rewrite(question)])[:k]

if LLM_READY:
    print(rewrite("why do models make things up"))
    for name, fn in [("hybrid", hybrid_search),
                     ("multi-query", multi_query_search),
                     ("RAG-Fusion", fusion_search)]:
        print(f"{name:12s} recall@5 = {recall_at_k(fn):.2f}   "
              f"MRR@10 = {mrr(fn):.2f}")

## Step 17 — Decomposition, step-back and HyDE 💳 API

[6. When the question is the problem](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-07-fable/06-when-the-question-is-the-problem/)

In [ ]:
def decompose_answer(question: str) -> str:
    subs = [
        re.sub(r"^\s*(?:[-•*]|\d+[.)])\s*", "", s).strip() for s in ask_llm(
            f"Split this question into 2–3 self-contained sub-questions, one "
            f"per line:\n{question}").splitlines() if s.strip()
    ]
    partial = []
    for sq in subs:
        a, _ = rag_answer(sq)
        partial.append(f"Q: {sq}\nA: {a}")
    return ask_llm(
        "Using only these partial answers, answer the original question. "
        "Keep the citations.\n\n" + "\n\n".join(partial) +
        f"\n\nOriginal question: {question}")

def step_back_search(question: str, k: int = 5):
    general = ask_llm(
        f"Rewrite this question as a more general one about "
        f"the underlying concept, "
        f"one line only:\n{question}")
    return rrf(hybrid_search(question, 20), hybrid_search(general, 20))[:k]

def hyde_search(question: str, k: int = 5):
    hypothetical = ask_llm(
        f"Write one paragraph, in the style of course notes, that would "
        f"answer:\n{question}")
    qv = embedder.encode([hypothetical], normalize_embeddings=True)[0]
    scores = chunk_vectors @ qv
    return [(float(scores[i]), int(i)) for i in np.argsort(-scores)[:k]]

if LLM_READY:
    print(
        decompose_answer(
            "How does DPO differ from RLHF, and what does the course say "
            "each one needs as training data?"))
    for name, fn in [("hybrid", hybrid_search),
                     ("step-back", step_back_search), ("HyDE", hyde_search)]:
        print(f"{name:10s} recall@5 = {recall_at_k(fn):.2f}   "
              f"MRR@10 = {mrr(fn):.2f}")

## Step 18 — Routing and query structuring (💳 API for logical routing and structuring)

[6. When the question is the problem](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-07-fable/06-when-the-question-is-the-problem/)

In [ ]:
import json

# Semantic routing: nearest destination description wins — no model call
ROUTES = {
    "notes": "questions about concepts, methods and examples explained in "
    "the course lectures",
    "homework": "questions about assignment rules, deadlines, grading "
    "criteria and submission format",
    "web": "questions about current events, prices, product releases or "
    "anything outside the course",
}
route_vectors = embedder.encode(list(ROUTES.values()),
                                normalize_embeddings=True)
def semantic_route(question: str) -> str:
    sims = route_vectors @ embedder.encode([question],
                                           normalize_embeddings=True)[0]
    return list(ROUTES)[int(np.argmax(sims))]

for q in [
        "what is a mixture of experts model",
        "how many chunks must the homework produce",
        "what did OpenAI release this week"
]:
    print(f"{semantic_route(q):9s} ← {q}")

# Logical routing: let the model classify against the same schema (structured
# output as JSON)
def logical_route(question: str) -> str:
    schema = "\n".join(f'- "{k}": {v}' for k, v in ROUTES.items())
    reply = ask_llm(
        f"Destinations:\n{schema}\n\nQuestion: {question}\n\n"
        f'Reply with JSON only: {{"destination": "<one of the keys>"}}')
    return json.loads(
        reply.strip().strip("`").removeprefix("json"))["destination"]

# Query structuring: text query + metadata filter, then a filtered Chroma
# search
def structured_search(question: str, k: int = 5):
    reply = ask_llm(
        f"Extract a search query and an optional lecture number (1–6) "
        f"from the question. "
        f'Reply with JSON only: {{"query": "...", "lecture": <int '
        f'or null>}}\n\nQuestion: {question}')
    # tolerate a ```json fence
    parsed = json.loads(reply.strip().strip("`").removeprefix("json"))
    where = {"lecture": parsed["lecture"]} if parsed.get("lecture") else None
    return parsed, chroma_search(parsed["query"], k=k, where=where)

if LLM_READY:
    print("logical:",
          logical_route("how many chunks must the homework produce"))
    parsed, hits = structured_search(
        "what did lecture 4 say about prompt injection?")
    print(parsed)
    for sim, meta, doc in hits[:3]:
        print(f'{sim:.3f}  L{meta["lecture"]} · {meta["heading"][:60]}')

## Step 19 — Search small, return large (💳 API for the summary index at the end)

[7. When the index is the problem](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-07-fable/07-when-the-index-is-the-problem/)

In [ ]:
section_by_id = {s["id"]: s for s in sections}

def parent_search(question: str, k: int = 3):
    """Rank chunks as before; return each chunk's whole section, without
    duplicates."""
    seen, out = set(), []
    for score, i in hybrid_search(question, 20):
        sid = chunks[i]["section"]
        if sid not in seen:
            seen.add(sid); out.append((score, section_by_id[sid]))
        if len(out) == k:
            break
    return out

q = ("Which worked example illustrates Least-to-Most prompting, and what are "
     "its three subquestions?")
chunk_ctx, _ = build_context(hybrid_search(q, 3), max_chars=10_000)
parent_ctx = "\n\n".join(
    f"[{n}] Lecture {s['lecture']} · {s['heading']}\n{s['text']}"
    for n, (_, s) in enumerate(parent_search(q), 1))
print("chunks:", len(chunk_ctx), "characters;  parents:", len(parent_ctx),
      "characters")
print("first parent:", parent_search(q)[0][1]["heading"])

# A summary index: embed a one-sentence summary per section, return the full
# section  💳 API
if LLM_READY:
    summaries = [
        ask_llm(f"Summarise in one sentence what question this passage "
                f"answers:\n\n{s['text'][:2000]}")
        # 40 of Lecture 1's 45 records keep the demo cheap
        for s in sections[:40]
    ]
    summary_vectors = embedder.encode(summaries, normalize_embeddings=True)
    def summary_search(question: str, k: int = 3):
        sims = summary_vectors @ embedder.encode([question],
                                                 normalize_embeddings=True)[0]
        return [(float(sims[i]), sections[i]) for i in np.argsort(-sims)[:k]]

    for sim, s in summary_search(
            "how does an image generator turn noise into a picture"):
        print(f"{sim:.3f}  L{s['lecture']} · {s['heading']}")

## Step 20 — Rerank, compress, and gate the answer (💳 API only for the query rewrite)

[8. Check before you answer](https://aiml-study-guides.onrender.com/en/generative-agentic-ai/lecture-07-fable/08-check-before-you-answer/)

In [ ]:
from sentence_transformers import CrossEncoder
# ~90 MB, downloaded once
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(question: str, results, k: int = 5):
    pairs = [(question, chunks[i]["text"]) for _, i in results]
    scores = reranker.predict(pairs)
    order = np.argsort(-scores)
    return [(float(scores[j]), results[j][1]) for j in order[:k]]

def rerank_search(question: str, k: int = 5):
    return rerank(question, hybrid_search(question, 30), k)

for name, fn in [("hybrid", hybrid_search), ("reranked", rerank_search)]:
    print(
        f"{name:9s} recall@5 = {recall_at_k(fn):.2f}   MRR@10 = {mrr(fn):.2f}")

def compress(question: str, text: str, keep: int = 3) -> str:
    """Keep the `keep` sentences most similar to the question, in their
    original order."""
    sents = [s for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]
    if len(sents) <= keep:
        return text
    vecs = embedder.encode([question] + sents, normalize_embeddings=True)
    sims = vecs[1:] @ vecs[0]
    top = sorted(np.argsort(-sims)[:keep])
    return " ".join(sents[i] for i in top)

q = "how does the model decide which tool to call"
best = chunks[rerank_search(q, 1)[0][1]]["text"]
print(len(best), "→", len(compress(q, best)), "characters\n",
      compress(q, best))

def gated_answer(question: str, threshold: float = 0.0, rounds: int = 2):
    """Corrective loop: answer only if the best reranked passage scores
    above the threshold; otherwise try one rewritten query; then give up
    explicitly."""
    tried = [question]
    for r in range(rounds):
        top = rerank_search(tried[-1], 5)
        if top[0][0] >= threshold:
            return (f"evidence ok (score {top[0][0]:.2f}) after {len(tried)} "
                    f"round(s) → would generate from {len(top)} passages")
        # last round, or no key: stop here
        if r == rounds - 1 or not LLM_READY:
            break
        # 💳 API
        tried.append(
            ask_llm(f"Rewrite this question using different vocabulary, one "
                    f"line:\n{tried[-1]}"))
    return (f"Not in the notes. (best score {top[0][0]:.2f} after "
            f"{len(tried)} round(s))")

print(gated_answer("why do models make things up"))
print(gated_answer("what does the course say about deploying with Kubernetes"))